# Delta Lake on Databricks

> **Prerequisites:** This notebook assumes you've read the Delta Lake notebooks in the `apache-spark` repo (`18-what-is-delta-lake`, `19-acid-time-travel`, `20-delta-operations-optimization`). Those cover the transaction log, ACID guarantees, time travel, OPTIMIZE, Z-ORDER, VACUUM, constraints, and CLONE in depth. This notebook covers the **Databricks-specific angle**: how Delta tables are registered and organised, multiple creation patterns, column mapping, liquid clustering, and deletion vectors.

In this notebook we'll cover:
- Managed vs external tables — the most important distinction for production
- The 3-level namespace: catalog, schema, table
- Four ways to create a Delta table and when to use each
- Column mapping — renaming and dropping columns without rewriting data
- Liquid clustering — Databricks's next-generation alternative to static Z-ORDER
- Deletion vectors — how Databricks speeds up deletes and updates

## Managed vs External Tables

This is the most exam-tested distinction in this section.

| | Managed Table | External Table |
|---|---|---|
| **Data location** | Managed by Databricks (default warehouse path) | You specify the `LOCATION` — data lives in your cloud storage |
| **DROP TABLE** | Deletes **both** metadata AND the data files | Deletes only the metadata — **data files remain** |
| **Ownership** | Databricks / Unity Catalog owns the lifecycle | You own the data; Databricks just points to it |
| **Use case** | Tables you create and manage inside Databricks | Pre-existing data in S3/ADLS, shared across tools, or data you need to survive DROP |

```sql
-- Managed: no LOCATION clause
CREATE TABLE orders (id INT, amount DOUBLE) USING DELTA;

-- External: LOCATION points to cloud storage
CREATE TABLE orders_ext (id INT, amount DOUBLE)
USING DELTA
LOCATION 's3://my-bucket/orders/';
```

> **Exam tip:** `DROP TABLE` on a **managed** table is destructive — data is gone. `DROP TABLE` on an **external** table is safe — the Parquet/Delta files remain on cloud storage. If a question asks how to remove a table definition without losing data, the answer is always an external table.

## The 3-Level Namespace

With **Unity Catalog**, Databricks uses a three-part naming convention:

```
catalog.schema.table
   │       │      │
   │       │      └── Table (or view, function, volume)
   │       └───────── Schema (= database)
   └───────────────── Catalog (top-level container)
```

**Example:**
```sql
SELECT * FROM prod_catalog.sales.orders;
```

Without Unity Catalog (legacy Hive metastore), only two levels exist: `schema.table` (what Spark calls a "database" is just a schema).

### Setting the Default Context

```sql
USE CATALOG prod_catalog;
USE SCHEMA  sales;

-- Now unqualified names resolve to prod_catalog.sales.*
SELECT * FROM orders;
```

### Catalog Types in Unity Catalog

| Catalog type | Purpose |
|---|---|
| **Regular** | Standard catalog for your data |
| **`hive_metastore`** | The legacy built-in catalog — backward compatible with pre-Unity Catalog workspaces |
| **Foreign** | Points to an external system (e.g., a Glue catalog, a PostgreSQL database via Lakehouse Federation) |
| **`__databricks_internal`** | System-managed, not user-accessible |

## Four Ways to Create a Delta Table

### 1. `CREATE TABLE` with DDL (empty schema)
Define columns explicitly. The table is empty until you insert data.

```sql
CREATE TABLE IF NOT EXISTS sales.orders (
  order_id    STRING NOT NULL,
  customer_id STRING,
  amount      DOUBLE,
  order_date  DATE
) USING DELTA
PARTITIONED BY (order_date)
COMMENT 'Cleaned orders from the silver layer';
```

### 2. `CREATE TABLE AS SELECT` (CTAS)
Creates a table and populates it in one step from a query. Schema is **inferred** from the query — you cannot specify column types.

```sql
CREATE OR REPLACE TABLE sales.orders_summary
USING DELTA
COMMENT 'Daily revenue by region'
AS
SELECT region, order_date, SUM(amount) AS revenue
FROM   sales.orders
GROUP BY region, order_date;
```

> `CTAS` does **not** support `PARTITIONED BY` with the standard syntax — use `CLUSTER BY` (liquid clustering) instead when you need clustering on a CTAS.

### 3. DataFrame `.saveAsTable()`
Writes a DataFrame and registers it as a table in the metastore. Creates a **managed** table by default.

```python
df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("sales.orders")
```

### 4. DataFrame `.save()` + register separately
Writes files to a path without registering in the metastore. Register later with `CREATE TABLE ... LOCATION`.

```python
# Step 1: write files
df.write.format("delta").mode("overwrite").save("s3://my-bucket/orders/")

# Step 2: register as external table
spark.sql("""
  CREATE TABLE IF NOT EXISTS sales.orders
  USING DELTA LOCATION 's3://my-bucket/orders/'
""")
```

This pattern is common when the data path already exists (e.g., Auto Loader wrote files before the table was registered).

## Column Mapping

By default, Delta Lake stores column names inside the Parquet files. Renaming or dropping a column means rewriting every file — expensive on large tables.

**Column mapping** decouples the logical column name (what SQL sees) from the physical column name (what's in the Parquet file). With it enabled, rename and drop operations are **metadata-only** — instant, zero data rewrite.

```sql
-- Enable column mapping on an existing table
ALTER TABLE sales.orders
SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion'   = '2',
  'delta.minWriterVersion'   = '5'
);

-- Now rename is instant
ALTER TABLE sales.orders RENAME COLUMN amount TO order_amount;

-- Drop is also instant (no data rewrite)
ALTER TABLE sales.orders DROP COLUMN legacy_field;
```

> Column mapping requires Delta reader version 2 and writer version 5. Older Spark clients that don't support these versions will be unable to read the table — plan upgrades before enabling.

## Liquid Clustering

Traditional `PARTITIONED BY` and `ZORDER BY` have limitations:
- Partitioning on high-cardinality columns creates millions of small directories
- Z-ORDER must be re-run with `OPTIMIZE` after every large write
- Changing the partition column requires a full table rewrite

**Liquid clustering** (introduced in Databricks Runtime 13.3) is a new approach that:
- Automatically co-locates related data as files are written and compacted
- Uses **Hilbert curves** to cluster on multiple columns efficiently
- Allows **incremental clustering** — only newly written data needs to be clustered on each `OPTIMIZE` run
- Lets you **change the clustering columns** without rewriting the table

```sql
-- Create with liquid clustering
CREATE TABLE sales.orders
CLUSTER BY (customer_id, order_date)
AS SELECT * FROM raw.orders;

-- Change clustering columns later (metadata only)
ALTER TABLE sales.orders CLUSTER BY (region, order_date);

-- OPTIMIZE still triggers compaction + clustering
OPTIMIZE sales.orders;
```

| | Partitioning | Z-ORDER | Liquid Clustering |
|---|---|---|---|
| **Good for** | Low-cardinality columns (e.g., date, region) | High-cardinality multi-column filters | Any cardinality, multi-column filters |
| **Overhead** | Directory explosion on high cardinality | Full re-cluster on every OPTIMIZE | Incremental — only new data clustered |
| **Change columns** | Full rewrite | Re-run OPTIMIZE | Metadata only |
| **CTAS support** | Limited | N/A | Yes (`CLUSTER BY` in CTAS) |

> Liquid clustering and `PARTITIONED BY` are mutually exclusive — you use one or the other, not both.

## Deletion Vectors

Normally, when you `DELETE` or `UPDATE` rows in a Delta table, Spark rewrites the entire Parquet file that contains those rows — even if only one row changed. On large files, this is expensive.

**Deletion vectors** change this: instead of rewriting the file, Delta records the deleted/updated row positions in a small sidecar file (`.dvbin`). Readers skip those rows without touching the data file.

```sql
-- Enable on an existing table
ALTER TABLE sales.orders
SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true');

-- Or set as default for all new tables in the session
SET spark.databricks.delta.properties.defaults.enableDeletionVectors = true;
```

**Effect:**
- `DELETE` and `UPDATE` become **dramatically faster** — write a tiny sidecar instead of rewriting a large file
- Reads are still correct — the reader merges the sidecar with the data file on the fly
- Eventually, `OPTIMIZE` rewrites files and removes the deletion vectors, fully cleaning up

| | Without Deletion Vectors | With Deletion Vectors |
|---|---|---|
| `DELETE` cost | Rewrites affected files | Writes a tiny sidecar file |
| Read overhead | None | Tiny (filter deleted rows) |
| Cleanup | Immediate (new files written) | On next `OPTIMIZE` run |
| Requires | Any Delta version | Delta 3.0+ / DBR 12.2+ |

## Hands-On: Managed vs External, Column Mapping, Liquid Clustering

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS delta_demo")
spark.sql("USE SCHEMA delta_demo")

In [ ]:
# Create a managed Delta table via CTAS
spark.sql("""
  CREATE OR REPLACE TABLE managed_orders
  USING DELTA
  COMMENT 'Managed table — DROP TABLE removes data'
  AS
  SELECT
    CAST(id AS STRING)        AS order_id,
    CONCAT('CUST', LPAD(CAST(id % 10 AS STRING), 3, '0')) AS customer_id,
    ROUND(RAND() * 500 + 10, 2) AS amount,
    DATE_ADD(CURRENT_DATE(), -(id % 90))  AS order_date,
    ARRAY('UK','US','DE','FR')[id % 4]    AS region
  FROM RANGE(100)
""")

spark.sql("DESCRIBE DETAIL managed_orders").select(
    "name", "location", "format", "numFiles", "sizeInBytes"
).show(truncate=False)

In [ ]:
# Enable column mapping, then rename a column
spark.sql("""
  ALTER TABLE managed_orders
  SET TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name',
    'delta.minReaderVersion'   = '2',
    'delta.minWriterVersion'   = '5'
  )
""")

# Rename is now instant — no data rewrite
spark.sql("ALTER TABLE managed_orders RENAME COLUMN amount TO order_amount")

spark.sql("DESCRIBE managed_orders").show()

In [ ]:
# Create a new table with liquid clustering
spark.sql("""
  CREATE OR REPLACE TABLE clustered_orders
  CLUSTER BY (customer_id, region)
  AS SELECT * FROM managed_orders
""")

# Trigger OPTIMIZE — clusters newly written data
spark.sql("OPTIMIZE clustered_orders")

# Inspect: clustering columns are stored in table properties
spark.sql("DESCRIBE DETAIL clustered_orders").select(
    "name", "clusteringColumns", "numFiles"
).show(truncate=False)

In [ ]:
# Change clustering columns — metadata only, no rewrite
spark.sql("ALTER TABLE clustered_orders CLUSTER BY (region, order_date)")

# Future OPTIMIZE runs will cluster on the new columns
print("Clustering columns changed — next OPTIMIZE will use (region, order_date)")

In [ ]:
# Enable deletion vectors and see the effect on a DELETE
spark.sql("""
  ALTER TABLE managed_orders
  SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true')
""")

spark.sql("DELETE FROM managed_orders WHERE region = 'DE'")

# History shows the DELETE wrote deletion vectors, not new data files
spark.sql("DESCRIBE HISTORY managed_orders").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(5, truncate=False)

In [ ]:
# Cleanup
spark.sql("DROP SCHEMA IF EXISTS delta_demo CASCADE")

## Quick Reference — Exam-Relevant Commands

```sql
-- Table inspection
DESCRIBE DETAIL my_table;          -- location, format, numFiles, sizeInBytes, clusteringColumns
DESCRIBE HISTORY my_table;         -- version, timestamp, operation, operationMetrics
DESCRIBE EXTENDED my_table;        -- schema + table properties + storage info

-- Optimisation
OPTIMIZE my_table;                          -- compaction
OPTIMIZE my_table ZORDER BY (col1, col2);   -- compaction + Z-ORDER (non-liquid tables)
VACUUM my_table RETAIN 168 HOURS;           -- delete files older than 7 days

-- Time travel
SELECT * FROM my_table VERSION AS OF 3;
SELECT * FROM my_table TIMESTAMP AS OF '2024-01-15';
RESTORE TABLE my_table TO VERSION AS OF 3;

-- Schema evolution
ALTER TABLE my_table ADD COLUMN notes STRING;
ALTER TABLE my_table RENAME COLUMN old_name TO new_name;  -- requires column mapping
ALTER TABLE my_table DROP COLUMN legacy_col;              -- requires column mapping

-- Constraints
ALTER TABLE my_table ADD CONSTRAINT pk_not_null CHECK (order_id IS NOT NULL);
ALTER TABLE my_table DROP CONSTRAINT pk_not_null;

-- Cloning
CREATE TABLE my_table_dev SHALLOW CLONE my_table;  -- metadata only, instant
CREATE TABLE my_table_bak DEEP    CLONE my_table;  -- full data copy
```

## Summary

- **Managed tables** let Databricks control the data lifecycle — `DROP TABLE` deletes both metadata and data files. **External tables** (with `LOCATION`) keep data files when the table is dropped.
- The **3-level namespace** (`catalog.schema.table`) is Unity Catalog's way of organising data assets. Without Unity Catalog, only `schema.table` exists.
- Delta tables can be created four ways: DDL `CREATE TABLE`, `CTAS`, DataFrame `.saveAsTable()`, or DataFrame `.save()` + `CREATE TABLE ... LOCATION`.
- **Column mapping** (`delta.columnMapping.mode = 'name'`) makes `RENAME COLUMN` and `DROP COLUMN` metadata-only operations — no data rewrite.
- **Liquid clustering** (`CLUSTER BY`) is the modern replacement for `PARTITIONED BY` + `ZORDER BY` — incremental, changeable without rewriting data, works well on any cardinality.
- **Deletion vectors** make `DELETE` and `UPDATE` fast by writing a small sidecar file instead of rewriting the affected Parquet files. `OPTIMIZE` cleans them up later.

Next: `04-querying-files-spark-sql.ipynb` — reading raw files directly with Spark SQL before they're registered as tables.